# Walmart Store Sales Forecasting — ARIMA multi-experiment + MLflow (via DagsHub)

In [ ]:
!pip install kaggle pmdarima statsmodels mlflow dagshub -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! chmod 600 /root/.kaggle/kaggle.json
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting
! unzip -o Walmart-Recruiting-Store-Sales-Forecasting.zip
! unzip -o train.csv.zip
! unzip -o test.csv.zip
! unzip -o features.csv.zip

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
from joblib import Parallel, delayed

from sklearn.base import BaseEstimator, TransformerMixin

import mlflow
import mlflow.pyfunc
import dagshub

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 110

## 1. MLflow / DagsHub setup

One MLflow experiment (`ARIMA_tarining`, spelled exactly as requested) holds all 5
hyperparameter experiments. Each experiment is a **parent run**; each of its four
pipeline stages (preprocessing, feature-engineering, feature-selection, training)
is logged as a **nested child run** under that parent.

In [ ]:
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

MLFLOW_EXPERIMENT_NAME = "ARIMA_tarining"  # kept exactly as specified
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

## 2. Load and merge data

In [ ]:
TRAIN_PATH    = "train.csv"
FEATURES_PATH = "features.csv"
STORES_PATH   = "stores.csv"

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=["Date"])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=["Date"])
stores_raw   = pd.read_csv(STORES_PATH)

df_merged = (
    train_raw
    .merge(features_raw, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores_raw,   on=["Store"],                       how="left")
)
df_merged = df_merged.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(f"Shape: {df_merged.shape}")
print(f"Store-Dept combinations: {df_merged.groupby(['Store','Dept']).ngroups}")

## 3. Global config

`N_SERIES` controls how many (Store, Dept) pairs are used in every experiment
(kept fixed across experiments so the 5 comparisons are apples-to-apples).
Set to `None` for a full run -- see the time-estimate discussion at the end.

In [ ]:
N_SERIES        = 30     # subset used for all 5 experiments; None = all ~3,331 pairs
VAL_START       = "2012-04-01"
H               = 4      # forecast horizon (weeks)
FOLD_WEEKS      = 12      # rolling-origin step size, same convention as the LightGBM notebook
MIN_HISTORY_WEEKS = 52    # minimum history required before the first fold

vol = df_merged.groupby(["Store", "Dept"])["Weekly_Sales"].mean().sort_values(ascending=False)
SERIES_PAIRS = list(vol.index[:N_SERIES]) if N_SERIES else list(vol.index)
print(f"Series used in every experiment: {len(SERIES_PAIRS)}")

## 4. Reusable building blocks (calendar/markdown exog, Fourier terms, per-series extraction)

In [ ]:
SUPERBOWL_DATES = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
LABORDAY_DATES  = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
THANKSGIVING_DATES = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
CHRISTMAS_DATES = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])


class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["Date"] = pd.to_datetime(X["Date"])
        week = X["Date"].dt.isocalendar().week.astype(int)
        X["week_sin"] = np.sin(2 * np.pi * week / 52)
        X["week_cos"] = np.cos(2 * np.pi * week / 52)
        X["is_thanksgiving"] = X["Date"].isin(THANKSGIVING_DATES).astype(int)
        X["is_christmas"]    = X["Date"].isin(CHRISTMAS_DATES).astype(int)
        X["is_superbowl"]    = X["Date"].isin(SUPERBOWL_DATES).astype(int)
        X["is_laborday"]     = X["Date"].isin(LABORDAY_DATES).astype(int)
        return X


class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        mdc = [c for c in ["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
               if c in X.columns]
        if mdc:
            X["total_markdown_log"] = np.log1p(X[mdc].fillna(0).sum(axis=1))
        return X


def fourier_terms(dates: pd.Series, period: int = 52, order: int = 3) -> pd.DataFrame:
    t = (dates - dates.min()).dt.days.values / 7.0
    out = {}
    for k in range(1, order + 1):
        out[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        out[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(out, index=dates.index)


BASE_EXOG_COLS = [
    "week_sin", "week_cos", "is_thanksgiving", "is_christmas",
    "is_superbowl", "is_laborday", "total_markdown_log", "IsHoliday",
]


def get_series(df: pd.DataFrame, store: int, dept: int, exog_cols: list) -> pd.DataFrame:
    s = (
        df[(df["Store"] == store) & (df["Dept"] == dept)]
        .sort_values("Date")
        .set_index("Date")
    )
    s = s.asfreq("W-FRI")
    s["Weekly_Sales"] = s["Weekly_Sales"].interpolate(limit_direction="both")
    for c in exog_cols:
        s[c] = s[c].ffill().bfill()
    return s


def make_rolling_folds(dates: pd.DatetimeIndex, val_start: str,
                        fold_weeks: int, min_history_weeks: int, h: int) -> list:
    """
    Rolling-origin folds: each fold's validation window starts `fold_weeks`
    earlier than the previous one, always leaving >= min_history_weeks of
    training history. Mirrors the FOLD_WEEKS=12 / MIN_HISTORY_WEEKS=52
    convention used for the tree-based models.
    """
    val_start_ts = pd.Timestamp(val_start)
    all_dates = dates[dates <= dates.max()]
    fold_starts = []
    cursor = val_start_ts
    min_start = dates.min() + pd.Timedelta(weeks=min_history_weeks)
    while cursor - pd.Timedelta(weeks=fold_weeks) >= min_start:
        cursor = cursor - pd.Timedelta(weeks=fold_weeks)
        fold_starts.append(cursor)
    fold_starts = sorted(set([val_start_ts] + fold_starts))
    folds = []
    for fs in fold_starts:
        train_mask = dates < fs
        val_mask   = (dates >= fs)
        if train_mask.sum() < min_history_weeks or val_mask.sum() == 0:
            continue
        folds.append(fs)
    return folds

## 5. Hyperparameter grid — 5 experiments

In [ ]:
HYPERPARAM_GRID = [
    {
        "name": "exp1_baseline",
        "fourier_order": 3, "exog_corr_threshold": 0.0,
        "max_p": 5, "max_q": 5, "max_d": 2,
        "information_criterion": "aic", "stepwise": True, "n_folds": 3,
    },
    {
        "name": "exp2_low_seasonality",
        "fourier_order": 1, "exog_corr_threshold": 0.0,
        "max_p": 5, "max_q": 5, "max_d": 2,
        "information_criterion": "aic", "stepwise": True, "n_folds": 3,
    },
    {
        "name": "exp3_high_seasonality",
        "fourier_order": 5, "exog_corr_threshold": 0.0,
        "max_p": 5, "max_q": 5, "max_d": 2,
        "information_criterion": "bic", "stepwise": True, "n_folds": 3,
    },
    {
        "name": "exp4_wide_order_search",
        "fourier_order": 3, "exog_corr_threshold": 0.0,
        "max_p": 8, "max_q": 8, "max_d": 2,
        "information_criterion": "aic", "stepwise": True, "n_folds": 3,
    },
    {
        "name": "exp5_filtered_exog",
        "fourier_order": 3, "exog_corr_threshold": 0.03,
        "max_p": 5, "max_q": 5, "max_d": 2,
        "information_criterion": "aicc", "stepwise": True, "n_folds": 3,
    },
]
for cfg in HYPERPARAM_GRID:
    print(cfg)

## 6. Pipeline stage functions

Each function does one job and returns what the next stage needs, so it can be
logged as its own MLflow run: `preprocessing -> feature_engineering ->
feature_selection -> training`.

In [ ]:
def stage_preprocessing(df_merged: pd.DataFrame, series_pairs: list, val_start: str):
    """Stage 1: validates series availability and logs data-level facts."""
    with mlflow.start_run(run_name="preprocessing", nested=True):
        mlflow.log_param("n_series", len(series_pairs))
        mlflow.log_param("val_start", val_start)
        mlflow.log_metric("rows_total", len(df_merged))
        mlflow.log_metric("date_min_epoch_days", (df_merged["Date"].min() - pd.Timestamp("1970-01-01")).days)
        mlflow.log_metric("date_max_epoch_days", (df_merged["Date"].max() - pd.Timestamp("1970-01-01")).days)
        pairs_df = pd.DataFrame(series_pairs, columns=["Store", "Dept"])
        pairs_df.to_csv("series_pairs.csv", index=False)
        mlflow.log_artifact("series_pairs.csv")
    return df_merged


def stage_feature_engineering(df_merged: pd.DataFrame, fourier_order: int):
    """Stage 2: builds calendar, markdown, and Fourier exogenous columns."""
    with mlflow.start_run(run_name="feature_engineering", nested=True):
        mlflow.log_param("fourier_order", fourier_order)

        calendar_tf = CalendarFeatureTransformer()
        markdown_tf = MarkdownFeatureTransformer()

        df_fe = calendar_tf.transform(df_merged)
        df_fe = markdown_tf.transform(df_fe)
        fcols = fourier_terms(df_fe["Date"], period=52, order=fourier_order)
        df_fe = pd.concat([df_fe, fcols], axis=1)

        exog_cols_full = BASE_EXOG_COLS + list(fcols.columns)
        mlflow.log_param("n_exog_candidates", len(exog_cols_full))
        mlflow.log_dict({"exog_candidates": exog_cols_full}, "exog_candidates.json")
    return df_fe, exog_cols_full


def stage_feature_selection(df_fe: pd.DataFrame, exog_cols_full: list,
                             series_pairs: list, corr_threshold: float):
    """
    Stage 3: drops exogenous regressors whose |Pearson correlation| with
    Weekly_Sales (pooled across the sampled series) falls below corr_threshold.
    corr_threshold=0.0 keeps everything -- used as a baseline/no-op in some experiments.
    """
    with mlflow.start_run(run_name="feature_selection", nested=True):
        mlflow.log_param("exog_corr_threshold", corr_threshold)

        idx = pd.MultiIndex.from_tuples(series_pairs, names=["Store", "Dept"])
        sample = df_fe[df_fe.set_index(["Store", "Dept"]).index.isin(idx)]
        corrs = sample[exog_cols_full + ["Weekly_Sales"]].corr()["Weekly_Sales"].drop("Weekly_Sales").abs()

        selected = list(corrs[corrs >= corr_threshold].index)
        dropped  = list(corrs[corrs < corr_threshold].index)

        mlflow.log_metric("n_selected_exog", len(selected))
        mlflow.log_metric("n_dropped_exog", len(dropped))
        mlflow.log_dict({"selected": selected, "dropped": dropped,
                          "correlations": corrs.round(4).to_dict()}, "feature_selection.json")
    return selected

## 7. Training stage — rolling-origin CV with `auto_arima`

In [ ]:
def fit_and_forecast_fold(df_fe, store, dept, exog_cols, fold_start, h,
                          max_p, max_q, max_d, information_criterion, stepwise):
    s = get_series(df_fe, store, dept, exog_cols)
    s_train = s[s.index < fold_start]
    s_val   = s[s.index >= fold_start].iloc[:h]

    if len(s_train) < 52 or s_val.empty:
        return None

    try:
        model = pm.auto_arima(
            s_train["Weekly_Sales"],
            exogenous=s_train[exog_cols] if exog_cols else None,
            seasonal=False,
            stepwise=stepwise,
            information_criterion=information_criterion,
            suppress_warnings=True,
            error_action="ignore",
            max_p=max_p, max_q=max_q, max_d=max_d,
        )
        preds = model.predict(
            n_periods=len(s_val),
            exogenous=s_val[exog_cols] if exog_cols else None,
        )
    except Exception:
        return None

    weights = np.where(s_val["IsHoliday"].values, 5.0, 1.0)
    wmae = float(np.sum(weights * np.abs(s_val["Weekly_Sales"].values - preds)) / np.sum(weights))
    return {"store": store, "dept": dept, "fold_start": str(fold_start.date()),
            "order": model.order, "wmae": wmae, "n_train": len(s_train)}


def stage_training(df_fe, exog_cols, series_pairs, val_start, fold_weeks,
                    min_history_weeks, h, cfg):
    """
    Stage 4: rolling-origin CV over `n_folds` folds x all series_pairs.
    Logs the mean CV WMAE as the experiment's headline metric.
    """
    with mlflow.start_run(run_name="training", nested=True) as run:
        for k in ["max_p", "max_q", "max_d", "information_criterion", "stepwise", "n_folds"]:
            mlflow.log_param(k, cfg[k])
        mlflow.log_param("exog_cols", exog_cols)

        dummy_dates = pd.date_range(df_fe["Date"].min(), df_fe["Date"].max(), freq="W-FRI")
        fold_starts = make_rolling_folds(dummy_dates, val_start, fold_weeks,
                                          min_history_weeks, h)[-cfg["n_folds"]:]
        mlflow.log_param("fold_starts", [str(f.date()) for f in fold_starts])

        t0 = time.time()
        all_results = Parallel(n_jobs=-1, verbose=0)(
            delayed(fit_and_forecast_fold)(
                df_fe, store, dept, exog_cols, fold_start, h,
                cfg["max_p"], cfg["max_q"], cfg["max_d"],
                cfg["information_criterion"], cfg["stepwise"],
            )
            for store, dept in series_pairs
            for fold_start in fold_starts
        )
        elapsed = time.time() - t0
        all_results = [r for r in all_results if r is not None]

        results_df = pd.DataFrame(all_results)
        cv_wmae = float(results_df["wmae"].mean()) if not results_df.empty else float("nan")

        mlflow.log_metric("cv_wmae", cv_wmae)
        mlflow.log_metric("n_fits_ok", len(results_df))
        mlflow.log_metric("n_fits_attempted", len(series_pairs) * len(fold_starts))
        mlflow.log_metric("elapsed_seconds", elapsed)

        results_df.to_csv("cv_results.csv", index=False)
        mlflow.log_artifact("cv_results.csv")

        print(f"[{cfg['name']}] cv_wmae={cv_wmae:.2f}  "
              f"({len(results_df)}/{len(series_pairs) * len(fold_starts)} fits ok, {elapsed:.1f}s)")
        return run.info.run_id, cv_wmae, results_df

## 8. Run all 5 experiments

In [ ]:
experiment_summaries = []

for cfg in HYPERPARAM_GRID:
    with mlflow.start_run(run_name=cfg["name"]) as parent_run:
        mlflow.log_params(cfg)

        df_pre = stage_preprocessing(df_merged, SERIES_PAIRS, VAL_START)
        df_fe, exog_candidates = stage_feature_engineering(df_pre, cfg["fourier_order"])
        selected_exog = stage_feature_selection(
            df_fe, exog_candidates, SERIES_PAIRS, cfg["exog_corr_threshold"]
        )
        train_run_id, cv_wmae, results_df = stage_training(
            df_fe, selected_exog, SERIES_PAIRS, VAL_START, FOLD_WEEKS,
            MIN_HISTORY_WEEKS, H, cfg,
        )

        mlflow.log_metric("cv_wmae", cv_wmae)  # duplicated at parent level for easy comparison

        experiment_summaries.append({
            "name": cfg["name"],
            "parent_run_id": parent_run.info.run_id,
            "training_run_id": train_run_id,
            "cv_wmae": cv_wmae,
            "fourier_order": cfg["fourier_order"],
            "exog_corr_threshold": cfg["exog_corr_threshold"],
            "selected_exog": selected_exog,
            "cfg": cfg,
        })

summary_df = pd.DataFrame(experiment_summaries).sort_values("cv_wmae")
display(summary_df[["name", "cv_wmae", "fourier_order", "exog_corr_threshold"]])

## 9. Select the best experiment

In [ ]:
best = min(experiment_summaries, key=lambda e: e["cv_wmae"])
print(f"Best experiment: {best['name']}  (cv_wmae={best['cv_wmae']:.2f})")
print(f"Hyperparameters: {best['cfg']}")
print(f"Selected exog columns: {best['selected_exog']}")

## 10. Register the best model

ARIMA/SARIMAX does not produce a single reusable set of weights the way a neural
net does -- each (Store, Dept) pair gets its own `auto_arima` fit. What we register
is a `pyfunc` wrapper that stores the **winning hyperparameters and selected exog
columns**, and fits/forecasts on demand for whichever (Store, Dept, history) it is
given. This mirrors the custom `PythonModel` wrapper pattern already used for the
DL models in this project.

In [ ]:
class ARIMAConfigForecaster(mlflow.pyfunc.PythonModel):
    """
    Stores the winning experiment's config. predict() expects a dict with
    'history_df' (columns: Date, Weekly_Sales, IsHoliday, + exog cols) and
    'horizon' (int weeks to forecast), fits auto_arima on the fly, and returns
    the forecast array. Fitting per-call is intentional: ARIMA has no fixed
    parameter vector to serialize per series ahead of time.
    """

    def __init__(self, cfg: dict, exog_cols: list):
        self.cfg = cfg
        self.exog_cols = exog_cols

    def predict(self, context, model_input):
        history_df = model_input["history_df"]
        horizon    = model_input.get("horizon", 4)
        history_df = history_df.set_index(pd.to_datetime(history_df["Date"])).asfreq("W-FRI")
        history_df["Weekly_Sales"] = history_df["Weekly_Sales"].interpolate(limit_direction="both")
        for c in self.exog_cols:
            history_df[c] = history_df[c].ffill().bfill()

        model = pm.auto_arima(
            history_df["Weekly_Sales"],
            exogenous=history_df[self.exog_cols] if self.exog_cols else None,
            seasonal=False,
            stepwise=self.cfg["stepwise"],
            information_criterion=self.cfg["information_criterion"],
            suppress_warnings=True,
            error_action="ignore",
            max_p=self.cfg["max_p"], max_q=self.cfg["max_q"], max_d=self.cfg["max_d"],
        )
        last_exog = history_df[self.exog_cols].iloc[[-1]] if self.exog_cols else None
        future_exog = (
            pd.concat([last_exog] * horizon, ignore_index=True) if last_exog is not None else None
        )
        return model.predict(n_periods=horizon, exogenous=future_exog)


REGISTERED_MODEL_NAME = "ARIMA_walmart_forecaster"

with mlflow.start_run(run_name=f"best_model_{best['name']}") as reg_run:
    mlflow.log_params(best["cfg"])
    mlflow.log_metric("cv_wmae", best["cv_wmae"])
    mlflow.log_param("selected_exog", best["selected_exog"])

    wrapper = ARIMAConfigForecaster(cfg=best["cfg"], exog_cols=best["selected_exog"])
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=wrapper,
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"Registered '{REGISTERED_MODEL_NAME}' from run {reg_run.info.run_id}")

## 11. Runtime notes

- ARIMA/SARIMAX (`statsmodels`/`pmdarima`) is CPU-bound; the T4 GPU sits idle for
  this whole notebook. What actually matters for speed is CPU core count and RAM
  of the Colab runtime, not the GPU.
- Cost driver: `n_fits = n_experiments (5) x n_folds x n_series`. Every fit is one
  `auto_arima` stepwise search.
- `N_SERIES` and `n_folds` are the two knobs that trade validation strength for
  runtime -- see the estimate below for concrete numbers on this dataset.